Idea:

Train NeuMF on bipartite dataset and predict ratings. Add predicted ratings to the metadata dataset and train an XGBoost with all the features for the final prediction.

Loss: Pairwise Losee
Metrics: RMSE, Precision@k, Recall@K

In [6]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from xgboost import XGBRegressor, XGBRanker
from sklearn.metrics import mean_squared_error

In [3]:
class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=32, mlp_dims=(64, 32, 16, 8)):
        super().__init__()

        # GMF part
        self.user_gmf = nn.Embedding(num_users, emb_dim)
        self.item_gmf = nn.Embedding(num_items, emb_dim)

        # MLP part
        self.user_mlp = nn.Embedding(num_users, emb_dim)
        self.item_mlp = nn.Embedding(num_items, emb_dim)

        mlp_layers = []
        input_dim = emb_dim * 2
        for d in mlp_dims:
            mlp_layers += [nn.Linear(input_dim, d), nn.ReLU()]
            input_dim = d
        self.mlp = nn.Sequential(*mlp_layers)

        # Final layer
        self.output = nn.Linear(emb_dim + mlp_dims[-1], 1)

    def forward(self, user_ids, item_ids):
        ug = self.user_gmf(user_ids)
        ig = self.item_gmf(item_ids)
        gmf_out = ug * ig

        um = self.user_mlp(user_ids)
        im = self.item_mlp(item_ids)
        mlp_out = self.mlp(torch.cat([um, im], dim=-1))

        x = torch.cat([gmf_out, mlp_out], dim=-1)
        pred = self.output(x).squeeze(-1)
        return pred

    def get_features(self, user_ids, item_ids):
        with torch.no_grad():
            ug = self.user_gmf(user_ids)
            ig = self.item_gmf(item_ids)
            um = self.user_mlp(user_ids)
            im = self.item_mlp(item_ids)

            gmf_out = ug * ig
            mlp_out = self.mlp(torch.cat([um, im], dim=-1))
            pred = self.output(torch.cat([gmf_out, mlp_out], dim=-1)).squeeze(-1)

            return {
                "user_gmf": ug,
                "item_gmf": ig,
                "user_mlp": um,
                "item_mlp": im,
                "gmf_out": gmf_out,
                "mlp_out": mlp_out,
                "pred": pred
            }

In [4]:
def train_neumf(model, train_loader, val_loader, epochs=10, lr=1e-3, device="cuda"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_func = nn.MSELoss()

    best_val_rmse = float("inf")
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for user_ids, item_ids, ratings in train_loader:
            user_ids = user_ids.to(device)
            item_ids = item_ids.to(device)
            ratings = ratings.float().to(device)

            optimizer.zero_grad()
            preds = model(user_ids, item_ids)
            loss = loss_func(preds, ratings)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(ratings)

        val_rmse = evaluate_rmse(model, val_loader, device=device)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch+1}: train_mse={total_loss/len(train_loader.dataset):.4f} val_rmse={val_rmse:.4f}")

    model.load_state_dict(best_state)
    return model

def evaluate_rmse(model, data_loader, device="cuda"):
    model.eval()
    preds_all = []
    y_all = []

    with torch.no_grad():
        for user_ids, item_ids, ratings in data_loader:
            user_ids = user_ids.to(device)
            item_ids = item_ids.to(device)
            ratings = ratings.float().to(device)

            preds = model(user_ids, item_ids)
            preds_all.append(preds.cpu())
            y_all.append(ratings.cpu())

    preds_all = torch.cat(preds_all)
    y_all = torch.cat(y_all)
    rmse = torch.sqrt(torch.mean((preds_all - y_all) ** 2)).item()
    return rmse